# Orthomosaic — Shared Configuration

**Shared configuration, imports, and helper functions for the Old Orchard photogrammetry example series.**

This notebook is `%run`-imported by notebooks 01–04 and the monitor. It installs the GeoBrix wheel, registers the **`exif_gbx`** reader and pyrx UDFs, and defines all SfM orchestration functions.

Two GeoBrix functions drive the non-product steps:
- **GeoBrix `exif_gbx`** — EXIF/GPS extraction at scale (replaces hand-rolled PIL loop).
- **GeoBrix `cog_gbx` writer** — COG conversion in the Spark tier (used in Part 3).

Product Databricks built-in functions are used where available:
- `ST_DistanceSphere` for geospatial pair filtering.
- `st_point` (Databricks native) for geometry column assembly.

> **Runtime.** Designed for **Serverless environment 5** — the [lightweight tier](https://databrickslabs.github.io/geobrix/docs/api/execution-tiers) (pure Python/PySpark, no JAR or GDAL init script). CPU sparse SfM only; dense reconstruction is Phase 2.

---

**Last Update:** September 21, 2026

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='line_magic_sanitizer')

In [ ]:
import csv, gc, json, os, shutil, sqlite3, stat, subprocess, tempfile, time
import urllib.request, warnings
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterator, Optional, Tuple

import numpy as np
import pandas as pd
import pycolmap
import rasterio
import requests
from PIL import Image as PILImage
from pyspark.sql import Window, functions as F, types as T
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, transform_bounds

# GeoBrix — lightweight tier (Serverless-safe, no JAR)
from databricks.labs.gbx.ds.register import register
from databricks.labs.gbx.pyrx import functions as rx
from databricks.labs.gbx import vizx as vz   # GeoBrix VizX rendering
from databricks.labs.gbx.pyrx.imagery import gsd_udf
register(spark)  # registers exif_gbx reader, gtiff_gbx / gtiff_gdal writers, pyrx UDFs

In [ ]:
# ── User Configuration ─────────────────────────────────────────────────────
# NADIR: nadir-weighted blending — each output pixel favors its most-nadir view
#   (sharper, but more seam-prone on relief); off = feather-average (smoother).
NADIR       = False

CATALOG_NAME   = "geospatial_docs"
SCHEMA_NAME    = "gopro"

# Image dataset — GitHub repository source
OWNER           = "Carlocktography"
REPO            = "sUAS_Photogrammetry_Suite_Test_Data"
BRANCH          = "trunk"
GH_DATASET_PATH = "datasets/OldOrchard_2017-07-22"
DOWNLOAD_DIR    = GH_DATASET_PATH.split("/")[-1]   # local driver-side folder

# Output directory (Databricks Workspace path, persists across sessions)
_current_user = spark.sql("SELECT current_user()").collect()[0][0]
output_dir    = f"/Volumes/geospatial_docs/orthomosaic/data/gopro/outputs/{DOWNLOAD_DIR}_result"
# COLMAP SfM scratch (SQLite DBs + sparse models) MUST live on LOCAL disk: SQLite
# raises "disk I/O error" on UC Volume FUSE (no locking/random-access). Ephemeral
# per run; only the final rasters below are written to the Volume.
SCRATCH_DIR   = f"/tmp/gbx_ortho_sfm/{DOWNLOAD_DIR}"

# UC Volume for PMTiles serving
PMTILES_VOLUME = "pmtiles"

# ── Camera / sensor parameters ─────────────────────────────────────────────
# Passed as `sensorWidthMm` / `focalLengthMm` overrides to exif_gbx when set.
# Set to None to trust EXIF-embedded values; override when EXIF is absent or wrong.
SENSOR_WIDTH_MM  = 6.17   # GoPro Hero-equivalent sensor width (mm); override as needed
FOCAL_LENGTH_MM  = None   # None = read from EXIF; e.g. 2.92 to override

# ── Geospatial pair-matching ───────────────────────────────────────────────
MAX_PAIR_DIST_M  = 30     # max GPS distance (m) to attempt pairwise matching

# ── Orthomosaic settings ───────────────────────────────────────────────────
GSD_CM           = None   # None = auto-derive from telemetry; or set e.g. 3.0
MAX_ORTHO_WORKERS = 16

# ── Group key (multi-flight support) ──────────────────────────────────────
# Set GROUP_KEY_COL to a column name in the exif_gbx metadata table to run
# SfM + ortho once per distinct value (e.g. "flight_date" derived from timestamp).
# The default constant group runs all images as a single orthomosaic.
GROUP_KEY_COL    = "_group"  # change to a real metadata column for multi-group runs

# ── GPS clustering (memory-bounded SfM) ────────────────────────────────
# Large surveys exceed single-node COLMAP mapper RAM, so images are partitioned
# into spatially-contiguous GPS clusters (each reconstructed separately, then
# georeferenced + blended into one orthomosaic). Lower TARGET_CLUSTER_IMAGES if a
# cluster still OOMs; raise it for fewer seams (up to the driver's limit).
# Cluster size is set by a MEMORY MODEL, not raw image count. Mapper RAM is
# driven by feature count ≈ images × MAX_IMAGE_SIZE^2 (every image is
# downsampled to MAX_IMAGE_SIZE before SIFT, so the source JPG size does NOT
# matter). The per-cluster image target therefore scales inversely with
# MAX_IMAGE_SIZE^2; 55 images was validated at 2000 px on the env5 driver.
# (Keep MAX_IMAGE_SIZE in sync with extract_features_to_df.)
MAX_IMAGE_SIZE        = 2000  # SIFT input cap (quality vs per-image RAM)
MIN_CLUSTER_IMAGES    = 8     # merge clusters smaller than this
CLUSTER_OVERLAP_FRAC  = 0.30  # shared boundary fraction (ties + blends neighbors)
_TARGET_AT_2000       = 55    # validated per-cluster image budget at 2000 px
TARGET_CLUSTER_IMAGES = max(MIN_CLUSTER_IMAGES,
                           int(_TARGET_AT_2000 * (2000.0 / MAX_IMAGE_SIZE) ** 2))
# Ortho blend sharpness: nadir weight ^ BLEND_GAMMA concentrates each pixel on the
# most-nadir view, cutting the parallax/averaging softness of flat-plane back-
# projection (higher = sharper, more seam-prone). Dense MVS (GPU) is the real fix.
BLEND_GAMMA           = 4.0 if NADIR else 1.0  # nadir-weighting exponent (see NADIR)

# ── Re-runnability guards ─────────────────────────────────────────────────
FORCE_DOWNLOAD   = False   # True = re-fetch JPEG dataset from GitHub
FORCE_RELOAD     = False   # True = rebuild all SfM tables (else reuse existing)

# ── Checkpoint FORCE flags (recovery / re-run) ─────────────────────────────
# Each long stage skips a cluster/group whose checkpoint signature matches AND
# whose output already exists on the Volume. Set the matching flag True to force
# that stage to recompute even when checkpointed. Changing a stage's knobs also
# forces recompute automatically (the signature changes).
FORCE_DENSE     = False   # nb1b: dense patch_match/fuse per cluster
FORCE_MOSAIC    = False   # nb1a: sparse image mosaic per group
FORCE_CORRECTED = False   # nb2: color correction per group
FORCE_COG       = False   # nb3-cog: Cloud-Optimised GeoTIFF per group
FORCE_PMTILES   = False   # nb3-pmtiles: PMTiles per group

# ── Per-group output paths ─────────────────────────────────────────────────
# Every artifact is namespaced by group key so multiple orthomosaics never
# collide. Deliverables (ortho, corrected, COG, PMTiles) land on the Volume under
# group_<grp>/; random-access intermediates (SfM SQLite + sparse, reprojected
# TIFF, working PMTiles/MBTiles) stay on local disk under the same group_<grp>/.
Path(SCRATCH_DIR).mkdir(parents=True, exist_ok=True)

def group_paths(grp):
    """All artifact paths for one group: Volume deliverables + local scratch."""
    _base = f"{output_dir}/group_{grp}"    # Volume (durable deliverables)
    _scr  = f"{SCRATCH_DIR}/group_{grp}"   # local disk (random-access intermediates)
    return {
        "group":         grp,
        "sfm_dir":       _scr,                               # COLMAP SQLite DBs + sparse (local)
        "sfm_persist":   f"{_base}/sfm",                     # durable sparse+GPS on Volume (FORCE_RELOAD reuse)
        "checkpoint":    f"{_base}/_checkpoint.json",   # per-group recovery manifest (Volume)
        "ortho":         f"{_base}/orthomosaic.tif",
        "dense_ortho": f"{_base}/orthomosaic_dense.tif",
        "dense_cloud": f"{_base}/dense.laz",
        "dense_dsm":   f"{_base}/dsm_dense.tif",
        "corrected":     f"{_base}/orthomosaic_corrected.tif",
        "cog_dir":       f"{_base}/cog",
        "cog":           f"{_base}/cog/orthomosaic_cog.tif",
        "cog_3857":      f"{_scr}/orthomosaic_cog_3857.tif",  # local (rasterio.warp seeks)
        "pmtiles_local": f"{_scr}/orthomosaic.pmtiles",       # local working (mbtiles=SQLite)
        "pmtiles":       f"{_base}/orthomosaic.pmtiles",      # Volume (final)
    }

def dense_cluster_paths(grp, cid):
    """Per-(group, cluster) dense deliverables on the Volume, named
    ``<product>_<group>_<cluster>`` so clusters never clobber each other. The
    MERGED, co-registered products keep the group-level names in group_paths()
    (dense_ortho / dense_dsm / dense_cloud)."""
    _base = f"{output_dir}/group_{grp}"
    return {
        "ortho": f"{_base}/orthomosaic_dense_{grp}_{cid}.tif",
        "dsm":   f"{_base}/dsm_dense_{grp}_{cid}.tif",
        "cloud": f"{_base}/dense_{grp}_{cid}.laz",
    }


def discover_groups():
    """Group keys already produced under output_dir (for downstream notebooks)."""
    _d = Path(output_dir)
    if not _d.exists():
        return []
    return sorted(p.name[len("group_"):] for p in _d.glob("group_*") if p.is_dir())

def ortho_input(grp):
    """Ortho the downstream series (nb2/nb3) consumes: the dense ortho if nb1b
    produced one on the Volume, else nb1a's sparse ortho (auto-detected — no flag)."""
    from pathlib import Path
    gp = group_paths(grp)
    return gp["dense_ortho"] if Path(gp["dense_ortho"]).exists() else gp["ortho"]

# ── Monitor ───────────────────────────────────────────────────────────────
total_images = 171  # updated by 01a_sfm_orthomosaic after QC filtering

In [ ]:
spark.catalog.setCurrentCatalog(CATALOG_NAME)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.catalog.setCurrentDatabase(SCHEMA_NAME)

# Output storage on a UC Volume (executor-writable; /Workspace is read-only
# from Serverless executors, so Spark writers like gtiff_gbx fail there).
spark.sql("CREATE SCHEMA IF NOT EXISTS geospatial_docs.orthomosaic")
spark.sql("CREATE VOLUME IF NOT EXISTS geospatial_docs.orthomosaic.data")

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

session = requests.Session()
session.headers.update({"Accept": "application/vnd.github+json"})


def github_contents_url(path):
    return f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{path}?ref={BRANCH}"


def iter_repo_files(path):
    url = github_contents_url(path)
    r = session.get(url, timeout=60)
    r.raise_for_status()
    items = r.json()
    if isinstance(items, dict) and items.get("type") == "file":
        yield items
        return
    for item in items:
        if item["type"] == "file":
            yield item
        elif item["type"] == "dir":
            yield from iter_repo_files(item["path"])


def is_image(path):
    return Path(path).suffix.lower() in IMAGE_EXTS


def download_file(file_info, output_root, max_retries=3, force=False):
    """Download a single GitHub file with retry. Skips an already-present file
    (idempotent) unless force=True, which re-fetches and overwrites it."""
    rel_path = Path(file_info["path"]).relative_to(GH_DATASET_PATH)
    out_path = Path(output_root) / rel_path
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if out_path.exists() and not force:
        return  # already staged
    url = file_info.get("download_url")
    if not url:
        print(f"Skipping (no download_url): {file_info['path']}")
        return
    for attempt in range(1, max_retries + 1):
        try:
            with session.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(out_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            return
        except Exception as exc:
            if attempt == max_retries:
                raise
            print(f"[retry {attempt}/{max_retries}] {out_path.name}: {exc}")
            time.sleep(2 ** attempt)

In [ ]:
def extract_features_to_df(iterator: Iterator[pd.DataFrame], _use_gpu: bool = False) -> Iterator[pd.DataFrame]:
    import gc, os, shutil, sqlite3, subprocess, sys, tempfile
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from PIL import Image

    MAX_IMAGE_SIZE = 2000  # SIFT input cap (quality knob; unchanged)

    # Per-image subprocess isolation. pycolmap's native SIFT allocations are not
    # reclaimed by Python GC across the many images one long-lived Spark worker
    # processes, so worker RSS climbs until the worker is killed (OOM). Running
    # extraction in a short-lived child process lets the OS reclaim ALL native
    # memory on process exit, keeping worker RSS flat regardless of image count.
    # Full quality is preserved (identical FeatureExtractionOptions).
    _child = tempfile.NamedTemporaryFile(
        mode="w", suffix="_extract_child.py", delete=False
    )
    _child.write(
        "import os, sys\n"
        "os.environ['OMP_NUM_THREADS'] = '1'\n"
        "os.environ['MKL_NUM_THREADS'] = '1'\n"
        "from pathlib import Path\n"
        "import pycolmap\n"
        "img_path, db_path, work_dir, max_size, use_gpu = sys.argv[1:6]\n"
        "work = Path(work_dir); work.mkdir(parents=True, exist_ok=True)\n"
        "link = work / Path(img_path).name\n"
        "if not os.path.lexists(str(link)):\n"
        "    os.symlink(img_path, link)\n"
        "db = Path(db_path)\n"
        "if db.exists():\n"
        "    db.unlink()\n"
        "opts = pycolmap.FeatureExtractionOptions()\n"
        "opts.max_image_size = int(max_size)\n"
        "opts.num_threads = 1\n"
        "opts.use_gpu = bool(int(use_gpu))\n"
        "pycolmap.extract_features(db, work, extraction_options=opts)\n"
    )
    _child.close()
    child_script = _child.name

    try:
        for pdf in iterator:
            for _, row in pdf.iterrows():
                img_path = row["source"].replace("file:", "")
                img_name = Path(img_path).name
                worker_db  = Path(f"/tmp/ex_{img_name}.db")
                worker_dir = Path(f"/tmp/ed_{img_name}")
                worker_dir.mkdir(parents=True, exist_ok=True)
                if worker_db.exists():
                    worker_db.unlink()
                try:
                    # Isolate the native SIFT extraction in a child process so its
                    # memory is fully reclaimed when the child exits.
                    proc = subprocess.run(
                        [sys.executable, child_script, img_path,
                         str(worker_db), str(worker_dir), str(MAX_IMAGE_SIZE),
                         str(int(_use_gpu))],
                        capture_output=True, text=True, timeout=300,
                    )
                    if proc.returncode != 0 or not worker_db.exists():
                        print(
                            f"[extract] SKIP {img_name}: child rc={proc.returncode} "
                            f"{proc.stderr.strip()[-300:]}",
                            flush=True,
                        )
                        continue
                    conn = sqlite3.connect(worker_db)
                    kp_data   = conn.execute("SELECT rows, cols, data FROM keypoints").fetchone()
                    desc_data = conn.execute("SELECT data FROM descriptors").fetchone()
                    gps_row   = conn.execute(
                        "SELECT position, coordinate_system, position_covariance, gravity "
                        "FROM pose_priors LIMIT 1"
                    ).fetchone()
                    if gps_row and gps_row[0]:
                        _pos = np.frombuffer(gps_row[0], dtype=np.float64)
                        gps_pos  = bytes(gps_row[0]) if np.all(np.isfinite(_pos)) else None
                        gps_cs   = int(gps_row[1]) if gps_pos is not None else 0
                        gps_cov  = bytes(gps_row[2]) if gps_row[2] is not None else None
                        gps_grav = bytes(gps_row[3]) if gps_row[3] is not None else None
                    else:
                        gps_pos = None; gps_cs = 0; gps_cov = None; gps_grav = None
                    cam_row = conn.execute(
                        "SELECT model, width, height, params FROM cameras LIMIT 1"
                    ).fetchone()
                    if cam_row:
                        cam_model_val  = int(cam_row[0])
                        cam_w_val      = int(cam_row[1])
                        cam_h_val      = int(cam_row[2])
                        cam_params_val = bytes(cam_row[3])
                    else:
                        cam_model_val = 2; cam_w_val = 4000; cam_h_val = 3000; cam_params_val = None
                    if kp_data and desc_data:
                        with Image.open(img_path) as img:
                            w, h = img.size
                        yield pd.DataFrame([{
                            "source": row["source"],
                            "keypoints": kp_data[2],
                            "kp_rows": kp_data[0],
                            "kp_cols": kp_data[1],
                            "descriptors": desc_data[0],
                            "width": w,
                            "height": h,
                            "gps_pos":  gps_pos,
                            "gps_cs":   gps_cs,
                            "gps_cov":  gps_cov,
                            "gps_grav": gps_grav,
                            "cam_model": cam_model_val,
                            "cam_w":     cam_w_val,
                            "cam_h":     cam_h_val,
                            "cam_params": cam_params_val,
                        }])
                    conn.close()
                finally:
                    if worker_db.exists():
                        worker_db.unlink()
                    shutil.rmtree(worker_dir, ignore_errors=True)
                    gc.collect()
    finally:
        try:
            os.unlink(child_script)
        except OSError:
            pass

In [ ]:
def find_geo_pairs(df_qc):
    """Geospatial pair discovery via ST_DistanceSphere (Databricks built-in).

    Databricks advantage: ``ST_DistanceSphere`` pair filtering keeps pair count
    linear in image count — exhaustive matching would be O(n^2).
    """
    df_qc.createOrReplaceTempView("drone_metadata")
    pairs_df = spark.sql(f"""
        SELECT
            a.source AS src1,
            b.source AS src2,
            ST_DistanceSphere(a.gps_geom, b.gps_geom) AS dist
        FROM drone_metadata a
        JOIN drone_metadata b ON a.source < b.source
        WHERE ST_DistanceSphere(a.gps_geom, b.gps_geom) < {MAX_PAIR_DIST_M}
    """)
    print(f"Geospatial filtering reduced pairs to: {pairs_df.count()}")
    return pairs_df

In [ ]:
def image_ids_to_pair_id(image_id1, image_id2):
    if image_id1 > image_id2:
        image_id1, image_id2 = image_id2, image_id1
    return 2147483647 * image_id1 + image_id2


def match_pairs_dist(iterator):
    import pycolmap, sqlite3, numpy as np, pandas as pd
    from pathlib import Path

    for pdf in iterator:
        results = []
        for _, row in pdf.iterrows():
            pair_hash = abs(hash(row["src1"] + row["src2"]))
            db_path = Path(f"/tmp/p_{pair_hash}.db")
            dummy   = Path(f"/tmp/d_{pair_hash}")
            if db_path.exists():
                db_path.unlink()
            try:
                dummy.mkdir(exist_ok=True)
                pycolmap.extract_features(db_path, dummy)
                conn   = sqlite3.connect(db_path)
                cursor = conn.cursor()
                for t in ["cameras", "images", "keypoints", "descriptors", "two_view_geometries"]:
                    cursor.execute(f"DELETE FROM {t}")
                _cm  = int(row["cam_model"]) if row.get("cam_model") is not None else 2
                _cw  = int(row["cam_w"])     if row.get("cam_w")     is not None else 4000
                _ch  = int(row["cam_h"])     if row.get("cam_h")     is not None else 3000
                _cp  = bytes(row["cam_params"]) if row.get("cam_params") is not None else                        np.array([1733.30, _cw/2, _ch/2, 0.0], dtype=np.float64).tobytes()
                cursor.execute(
                    "INSERT INTO cameras (camera_id, model, width, height, params, prior_focal_length) "
                    "VALUES (1, ?, ?, ?, ?, 1)", (_cm, _cw, _ch, _cp))
                cursor.execute(
                    "INSERT INTO images (image_id, name, camera_id) VALUES (1, 'img1', 1), (2, 'img2', 1)")
                cursor.execute(
                    "INSERT INTO keypoints (image_id, rows, cols, data) VALUES (?, ?, ?, ?)",
                    (1, int(row["kp_rows_1"]), int(row["kp_cols_1"]), row["kp1"]))
                cursor.execute(
                    "INSERT INTO keypoints (image_id, rows, cols, data) VALUES (?, ?, ?, ?)",
                    (2, int(row["kp_rows_2"]), int(row["kp_cols_2"]), row["kp2"]))
                d1_rows = len(row["desc1"]) // 128
                d2_rows = len(row["desc2"]) // 128
                cursor.execute(
                    "INSERT INTO descriptors (image_id, rows, cols, data, type) VALUES (?, ?, 128, ?, 0)",
                    (1, d1_rows, row["desc1"]))
                cursor.execute(
                    "INSERT INTO descriptors (image_id, rows, cols, data, type) VALUES (?, ?, 128, ?, 0)",
                    (2, d2_rows, row["desc2"]))
                conn.commit(); conn.close()
                m_opts = pycolmap.FeatureMatchingOptions()
                m_opts.sift.max_ratio = 0.85
                pycolmap.match_exhaustive(database_path=db_path, matching_options=m_opts)
                conn = sqlite3.connect(db_path)
                match_row = conn.execute(
                    "SELECT rows, data, config FROM two_view_geometries LIMIT 1"
                ).fetchone()
                conn.close()
                if match_row and match_row[0] >= 10:
                    results.append({
                        "src1": row["src1"], "src2": row["src2"],
                        "matches": match_row[1], "match_config": match_row[2],
                    })
            except Exception as e:
                print(f"[match_pairs_dist] FAILED {row['src1']} <-> {row['src2']}: {e}")
            finally:
                if db_path.exists(): db_path.unlink()
                if dummy.exists(): dummy.rmdir()
        yield pd.DataFrame(results)

In [ ]:
def run_spark_sfm(
    df_qc,
    img_dir,
    output_dir,
    group_key=None,
    do_overwrite_features=False,
    do_overwrite_matches=False,
    do_overwrite_master=False,
    persist_dir=None,
    serialize_extract=False,
):
    """Distributed SfM pipeline for one group of drone images.

    Parameters
    ----------
    df_qc:
        Spark DF with 'source' column (full path to JPEGs) and optional GPS geometry.
    img_dir:
        POSIX path to the JPEG directory.
    output_dir:
        Persistent storage root for SfM artifacts.
    group_key:
        Optional string label for this group (used as a table/path suffix).
    do_overwrite_features, do_overwrite_matches, do_overwrite_master:
        Force re-extraction / re-matching / re-mapping even when tables exist.
    """
    _sfx = f"_{group_key}" if group_key else ""
    feat_tbl  = f"extracted_features{_sfx}"
    match_tbl = f"verified_matches{_sfx}"

    num_partitions = df_qc.count()
    # Memory-safe (Serverless): one image per mapInPandas Arrow batch, so a Python
    # worker holds at most a single image’s pycolmap working set (per-task RAM ~1-2 GB).
    # SIFT params/quality unchanged. Guarded per the Serverless spark.conf convention.
    try:
        spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1")
    except Exception:
        pass
    t0 = time.perf_counter()

    # ── Stage 1: Distributed feature extraction ───────────────────────────
    t1 = time.perf_counter()
    try:
        _table_ok = (
            spark.catalog.tableExists(feat_tbl)
            and "gps_pos" in spark.table(feat_tbl).columns
            and "cam_model" in spark.table(feat_tbl).columns
        )
        # Empty/stale cache must NOT be reused: 0 cached features yields 0 matches
        # downstream and misreads as a reconstruction failure - re-extract instead.
        _cached_n = spark.table(feat_tbl).count() if _table_ok else 0
        _reuse = _table_ok and _cached_n > 0 and not do_overwrite_features
        if _reuse:
            df_features = spark.table(feat_tbl)
            print(f"[Stage 1] reusing cached features: {_cached_n} images (FORCE_RELOAD=False)")
        else:
            if _table_ok and _cached_n == 0 and not do_overwrite_features:
                print("[Stage 1] cached features empty - rebuilding (extracting fresh)")
            feature_schema = (
                "source string, keypoints binary, kp_rows int, kp_cols int, "
                "descriptors binary, width int, height int, "
                "gps_pos binary, gps_cs int, gps_cov binary, gps_grav binary, "
                "cam_model int, cam_w int, cam_h int, cam_params binary"
            )
            # serialize_extract: force a single partition so extraction runs one
            # SIFT child at a time (no concurrent SIFT on a node). Retry-loop
            # fallback after worker OOMs; slower but memory-bounded. SIFT quality
            # unchanged. AQE does NOT reliably honor a bare repartition(N) /
            # repartition(1); the respected idioms are repartition(N, col) for
            # fan-out (one task per image, keyed on the unique source path) and
            # coalesce(1) to force a single partition for the serialized fallback.
            if serialize_extract:
                print("[Stage 1] serialized extraction (coalesce 1) - memory-bounded OOM fallback")
            _extract_df = (
                df_qc.select("source").coalesce(1)
                if serialize_extract
                else df_qc.select("source").repartition(num_partitions, "source")
            )
            (
                _extract_df
                .mapInPandas(extract_features_to_df, schema=feature_schema)
                .write.mode("overwrite").option("overwriteSchema", "true")
                .saveAsTable(feat_tbl)
            )
            df_features = spark.table(feat_tbl)
            print(f"[Stage 1] extracted features: {time.perf_counter()-t1:.1f}s | {df_features.count()} images")
    except Exception as e:
        print(f"[ERROR Stage 1] Feature extraction failed after {time.perf_counter()-t1:.1f}s: {e}")
        raise RuntimeError(f"[Stage 1 extract] {e}") from e

    # ── Stage 2–3: Spatial pair discovery + distributed matching ─────────
    t2 = time.perf_counter()
    match_count = 0
    df_verified_matches = None
    try:
        if not spark.catalog.tableExists(match_tbl) or do_overwrite_matches:
            df_pairs = find_geo_pairs(df_qc)
            (
                df_pairs
                .join(df_features.alias("f1"), df_pairs.src1 == F.col("f1.source"))
                .join(df_features.alias("f2"), df_pairs.src2 == F.col("f2.source"))
                .select(
                    "src1", "src2",
                    F.col("f1.keypoints").alias("kp1"),
                    F.col("f1.kp_rows").alias("kp_rows_1"),
                    F.col("f1.kp_cols").alias("kp_cols_1"),
                    F.col("f1.descriptors").alias("desc1"),
                    F.col("f2.keypoints").alias("kp2"),
                    F.col("f2.kp_rows").alias("kp_rows_2"),
                    F.col("f2.kp_cols").alias("kp_cols_2"),
                    F.col("f2.descriptors").alias("desc2"),
                    F.col("f1.cam_model").alias("cam_model"),
                    F.col("f1.cam_w").alias("cam_w"),
                    F.col("f1.cam_h").alias("cam_h"),
                    F.col("f1.cam_params").alias("cam_params"),
                )
                .repartition(num_partitions)
                .mapInPandas(
                    match_pairs_dist,
                    schema="src1 string, src2 string, matches binary, match_config int",
                )
                .write.mode("overwrite").option("overwriteSchema", "true")
                .saveAsTable(match_tbl)
            )
        df_verified_matches = spark.table(match_tbl)
        match_count = df_verified_matches.count()
        print(f"[Stage 2-3] Pair matching: {time.perf_counter()-t2:.1f}s | {match_count} verified pairs")
    except Exception as e:
        print(f"[ERROR Stage 2-3] Matching failed after {time.perf_counter()-t2:.1f}s: {e}")
        raise RuntimeError(f"[Stage 2-3 match] {e}") from e

    if match_count == 0:
        _img_n = df_features.count()
        if _img_n == 0:
            raise RuntimeError(
                "Stage 1: 0 features extracted - empty/failed extraction "
                "(if FORCE_RELOAD=False, set True to rebuild the feature cache)."
            )
        raise RuntimeError(
            f"Stage 3: 0 matches from {_img_n} images - check image overlap "
            "or descriptor extraction."
        )

    # ── GPS prior validation ───────────────────────────────────────────────
    # Warn (don't hard-fail) if fewer than 3 images have GPS — incremental
    # mapping will still attempt registration but without GPS-constrained BA.
    gps_check_count = df_features.filter(F.col("gps_pos").isNotNull()).count()
    if gps_check_count < 3:
        print(
            f"[WARN] Only {gps_check_count} images have valid GPS priors "
            "(need ≥3 for GPS-constrained BA). Proceeding without GPS constraints."
        )
    else:
        print(f"  GPS priors available: {gps_check_count} images")

    # ── Stage 4–5: Master DB assembly + incremental mapping ───────────────
    t4 = time.perf_counter()
    try:
        import sqlite3 as _sq, numpy as np, shutil as _shutil
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        master_db  = Path(output_dir) / f"master_sfm{_sfx}.db"
        sparse_dir = Path(output_dir) / "sparse"
        snap_dir   = Path(output_dir) / "snapshots"

        # Durable reuse (FORCE_RELOAD=False): a prior run persists the sparse model
        # (.bin — safe to copy) and GPS priors (JSON) to persist_dir on the Volume.
        # SfM's SQLite master DB cannot live on a Volume and local scratch is ephemeral
        # per job, so cross-run reuse keys on these durable artifacts, not master_db.
        _psparse   = Path(persist_dir) / "sparse" if persist_dir else None
        _pgps      = Path(persist_dir) / "gps_priors.json" if persist_dir else None
        _gps_local = Path(output_dir) / "gps_priors.json"

        def _gps_from_master(db):
            import sqlite3 as __sq
            _c = __sq.connect(str(db)); _o = {}
            for _n, _p in _c.execute("SELECT i.name, p.position FROM images i "
                                     "JOIN pose_priors p ON p.pose_prior_id = i.image_id"):
                _o[_n] = np.frombuffer(_p, dtype=np.float64).tolist()
            _c.close(); return _o

        if (not do_overwrite_master and _psparse is not None
                and _psparse.exists() and _pgps.exists()):
            if sparse_dir.exists(): _shutil.rmtree(sparse_dir)
            _shutil.copytree(_psparse, sparse_dir)
            _shutil.copy2(_pgps, _gps_local)
            print(f"[Stage 4-5] Reusing persisted SfM (FORCE_RELOAD=False) ← {persist_dir} "
                  f"({time.perf_counter()-t0:.0f}s)")
            return {"sparse_dir": str(sparse_dir), "gps_json": str(_gps_local),
                    "models": None, "registered": None, "reused": True}

        if not master_db.exists() or do_overwrite_master:
            if master_db.exists():
                master_db.unlink()

            with tempfile.TemporaryDirectory() as _init_tmp:
                _schema_opts = pycolmap.FeatureExtractionOptions()
                _schema_opts.num_threads = 1
                pycolmap.extract_features(master_db, Path(_init_tmp), extraction_options=_schema_opts)

            local_features = df_features.collect()
            local_matches  = df_verified_matches.collect()

            conn   = _sq.connect(master_db)
            cursor = conn.cursor()

            local_features_sorted = sorted(local_features, key=lambda r: Path(r.source).name)
            first_feat = local_features_sorted[0]
            cam_model_id     = int(first_feat.cam_model)    if first_feat.cam_model  is not None else 2
            cam_w_native     = int(first_feat.cam_w)        if first_feat.cam_w      is not None else 4000
            cam_h_native     = int(first_feat.cam_h)        if first_feat.cam_h      is not None else 3000
            cam_params_bytes = bytes(first_feat.cam_params) if first_feat.cam_params is not None else                 np.array([1733.30, cam_w_native/2, cam_h_native/2, 0.0], dtype=np.float64).tobytes()
            cursor.execute("DELETE FROM cameras")
            cursor.execute(
                "INSERT INTO cameras (camera_id, model, width, height, params, prior_focal_length) "
                "VALUES (1, ?, ?, ?, ?, 1)",
                (cam_model_id, cam_w_native, cam_h_native, cam_params_bytes))
            cursor.execute(
                "INSERT INTO rigs (rig_id, ref_sensor_id, ref_sensor_type) VALUES (1, 1, 0)")

            img_path_to_id = {}
            has_gps = False
            for row in local_features_sorted:
                img_name = Path(row.source).name
                cursor.execute("INSERT INTO images (name, camera_id) VALUES (?, 1)", (img_name,))
                image_id = cursor.lastrowid
                img_path_to_id[row.source] = image_id
                cursor.execute("INSERT INTO frames (frame_id, rig_id) VALUES (?, 1)", (image_id,))
                cursor.execute(
                    "INSERT INTO frame_data (frame_id, data_id, sensor_id, sensor_type) "
                    "VALUES (?, ?, 1, 0)", (image_id, image_id))
                cursor.execute(
                    "INSERT INTO keypoints (image_id, rows, cols, data) VALUES (?, ?, ?, ?)",
                    (image_id, row.kp_rows, row.kp_cols, row.keypoints))
                desc_rows = len(row.descriptors) // 128
                cursor.execute(
                    "INSERT INTO descriptors (image_id, rows, cols, data, type) VALUES (?, ?, 128, ?, 0)",
                    (image_id, desc_rows, row.descriptors))
                gps_pos_bytes = bytes(row.gps_pos) if row.gps_pos else None
                if gps_pos_bytes and len(gps_pos_bytes) == 24:
                    _pos = np.frombuffer(gps_pos_bytes, dtype=np.float64)
                    if np.all(np.isfinite(_pos)):
                        gps_cov_bytes  = bytes(row.gps_cov)  if row.gps_cov  else np.full(9, np.nan, dtype=np.float64).tobytes()
                        gps_grav_bytes = bytes(row.gps_grav) if row.gps_grav else np.array([0.0, 1.0, 0.0], dtype=np.float64).tobytes()
                        cursor.execute(
                            "INSERT INTO pose_priors "
                            "(pose_prior_id, corr_data_id, corr_sensor_id, corr_sensor_type, "
                            " position, coordinate_system, position_covariance, gravity) "
                            "VALUES (?, ?, 1, 0, ?, ?, ?, ?)",
                            (image_id, image_id, gps_pos_bytes, int(row.gps_cs), gps_cov_bytes, gps_grav_bytes))
                        has_gps = True

            best_match_count = 0
            best_init_id1 = best_init_id2 = -1
            for row in local_matches:
                id1 = img_path_to_id[row.src1]
                id2 = img_path_to_id[row.src2]
                match_data = bytes(row.matches)
                if id1 > id2:
                    id1, id2 = id2, id1
                    arr = np.frombuffer(match_data, dtype=np.uint32).reshape(-1, 2)
                    match_data = arr[:, ::-1].astype(np.uint32).tobytes()
                pair_id    = image_ids_to_pair_id(id1, id2)
                match_rows = len(match_data) // 8
                cursor.execute(
                    "INSERT INTO two_view_geometries (pair_id, rows, cols, data, config) "
                    "VALUES (?, ?, 2, ?, ?)", (pair_id, match_rows, match_data, row.match_config))
                if match_rows > best_match_count:
                    best_match_count = match_rows
                    best_init_id1, best_init_id2 = id1, id2
            conn.commit(); conn.close()
            print(f"  Master DB: {len(local_features_sorted)} images, "
                  f"GPS={has_gps}, best init pair: {best_init_id1}/{best_init_id2} "
                  f"({best_match_count} verified matches)")

            # Incremental mapping
            local_map_db = Path("/tmp/master_sfm_map.db")
            if local_map_db.exists():
                local_map_db.unlink()
            _shutil.copy2(master_db, local_map_db)

            if sparse_dir.exists(): _shutil.rmtree(sparse_dir)
            if snap_dir.exists():   _shutil.rmtree(snap_dir)
            sparse_dir.mkdir(parents=True, exist_ok=True)
            snap_dir.mkdir(parents=True, exist_ok=True)

            mapper_opts = pycolmap.IncrementalPipelineOptions()
            mapper_opts.mapper.init_min_tri_angle      = 3.0
            mapper_opts.mapper.init_max_forward_motion = 0.99
            mapper_opts.mapper.init_min_num_inliers    = 50
            mapper_opts.mapper.abs_pose_min_num_inliers  = 15
            mapper_opts.mapper.abs_pose_min_inlier_ratio = 0.1
            mapper_opts.ba_global_max_num_iterations   = 15
            mapper_opts.ba_local_max_num_iterations    = 15
            mapper_opts.init_num_trials                = 50
            mapper_opts.max_runtime_seconds            = 1800
            mapper_opts.snapshot_frames_freq = 10
            mapper_opts.snapshot_path        = snap_dir
            mapper_opts.use_prior_position                = has_gps
            mapper_opts.use_robust_loss_on_prior_position = has_gps
            mapper_opts.init_image_id1 = best_init_id1
            mapper_opts.init_image_id2 = best_init_id2
            mapper_opts.image_names = [Path(r.source).name for r in local_features_sorted]

            _registered = [0]
            _t_map = [time.perf_counter()]

            def _on_init():
                print(f"[Mapping] Initial pair found — registering remaining images...", flush=True)

            def _on_next():
                _registered[0] += 1
                elapsed = time.perf_counter() - _t_map[0]
                rate    = _registered[0] / elapsed if elapsed > 0 else 0
                eta     = (len(local_features_sorted) - _registered[0]) / rate if rate > 0 else float("inf")
                print(
                    f"[Mapping] {_registered[0]}/{len(local_features_sorted)} images "
                    f"({100*_registered[0]/len(local_features_sorted):.0f}%) | "
                    f"elapsed {elapsed:.0f}s | ETA ~{eta:.0f}s",
                    flush=True,
                )

            print(f"Running incremental mapping ({len(local_features_sorted)} images)...", flush=True)
            reconstructions = pycolmap.incremental_mapping(
                database_path=local_map_db,
                image_path=Path(img_dir),
                output_path=sparse_dir,
                options=mapper_opts,
                initial_image_pair_callback=_on_init,
                next_image_callback=_on_next,
            )
            local_map_db.unlink(missing_ok=True)
            if snap_dir.exists():
                _shutil.rmtree(snap_dir)
            elapsed_total = time.perf_counter() - t4
            print(
                f"[Stage 4-5] SfM complete: {len(reconstructions)} model(s) in "
                f"{elapsed_total:.0f}s ({_registered[0]}/{len(local_features_sorted)} images registered)."
            )
            import json as __json
            with open(_gps_local, "w") as _f:
                __json.dump(_gps_from_master(master_db), _f)
            if persist_dir:
                Path(persist_dir).mkdir(parents=True, exist_ok=True)
                if _psparse.exists(): _shutil.rmtree(_psparse)
                _shutil.copytree(sparse_dir, _psparse)
                _shutil.copy2(_gps_local, _pgps)
                print(f"[Stage 4-5] Persisted SfM → {persist_dir} (reusable on FORCE_RELOAD=False)")
            return {
                "sparse_dir": str(sparse_dir),
                "gps_json":   str(_gps_local),
                "models":     len(reconstructions),
                "registered": _registered[0],
            }
        elapsed_total = time.perf_counter() - t0
        print(f"[Stage 4-5] Pre-existing master DB (do_overwrite_master=False). "
              f"Run monitor notebook for status. ({elapsed_total:.0f}s)")
        import json as __json
        with open(_gps_local, "w") as _f:
            __json.dump(_gps_from_master(master_db), _f)
        return {"sparse_dir": str(sparse_dir), "gps_json": str(_gps_local), "models": None, "registered": None}
    except Exception as e:
        print(f"[ERROR Stage 4-5] SfM failed after {time.perf_counter()-t4:.1f}s: {e}")
        raise

In [ ]:
# ── Dense MVS (GPU, nb1b Phase 2b) ─────────────────────────────────────────
# COLMAP dense multi-view stereo on ONE cluster's sparse model + undistorted images:
# undistort_images -> patch_match_stereo (GPU) -> stereo_fusion -> fused.ply (colored
# dense point cloud). Runs in a single per-cluster CUDA context (unlike sparse, which is
# distributed CPU); this is the GPU-bound payoff. Requires pycolmap-cuda12 (env5 GPU).
def run_dense_mvs(sparse_dir, image_dir, work_dir, max_image_size=None):
    """Dense MVS for one cluster; returns the path to fused.ply (colored point cloud).

    max_image_size bounds patch-match memory (A10 23 GB); None = COLMAP default.
    """
    import pycolmap
    from pathlib import Path
    work = Path(work_dir); work.mkdir(parents=True, exist_ok=True)
    # COLMAP writes the reconstruction into a numbered subdir (sparse/0, ...), so
    # cameras.bin may be under sparse/<n>/ rather than sparse/. Resolve the model dir.
    sp = Path(sparse_dir)
    if (sp / "cameras.bin").exists() or (sp / "cameras.txt").exists():
        model_dir = sp
    else:
        _cands = [d for d in sorted(sp.iterdir()) if d.is_dir()
                  and ((d / "cameras.bin").exists() or (d / "cameras.txt").exists())]
        if not _cands:
            raise RuntimeError(f"run_dense_mvs: no COLMAP model under {sparse_dir}")
        model_dir = max(_cands, key=lambda d: (d / "images.bin").stat().st_size
                        if (d / "images.bin").exists() else 0)
    # 1) Undistort the cluster's images against its sparse model into a dense workspace.
    pycolmap.undistort_images(output_path=str(work), input_path=str(model_dir), image_path=str(image_dir))
    # 2) GPU patch-match stereo (per-pixel depth/normal maps).
    pm = pycolmap.PatchMatchOptions()
    if max_image_size:
        pm.max_image_size = int(max_image_size)
    pycolmap.patch_match_stereo(str(work), options=pm)
    # 3) Fuse depth maps into a colored dense point cloud (binary PLY).
    ply = str(work / "fused.ply")
    # output_type="PLY" writes a binary .ply point cloud to output_path. The
    # default "bin" instead writes a COLMAP model DIRECTORY there and reads it
    # back, which fails ExistsDir when output_path is a .ply file (pycolmap 4.2).
    pycolmap.stereo_fusion(output_path=ply, workspace_path=str(work), output_type="PLY")
    if not Path(ply).exists():
        raise RuntimeError(f"run_dense_mvs: stereo_fusion produced no fused.ply at {ply}")
    return ply

In [ ]:
# --- GPS-cluster orthomosaic: place every cluster in ONE ENU frame, tying
# --- neighbours through their SHARED overlap cameras (not independent GPS fits),
# --- then blend all clusters' projections into a single canvas. Peak RAM = one
# --- cluster reconstruction at a time + the output canvas.

def _umeyama_sim3(src, dst):
    """Least-squares similarity (scale, R, t) mapping src -> dst (Umeyama)."""
    n = len(src)
    mu_s, mu_d = src.mean(0), dst.mean(0)
    sc, dc = src - mu_s, dst - mu_d
    var_s = (sc ** 2).sum() / n
    cov = (dc.T @ sc) / n
    U, d, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    scale = (d * S.diagonal()).sum() / var_s
    t = mu_d - scale * R @ mu_s
    return scale, R, t


def _read_gps_priors(gps_json):
    """{image_name: np.array([lat, lon, alt])} from a persisted GPS-priors JSON.

    GPS priors are persisted as JSON (not read from the SQLite master DB) so the
    orthomosaic step works from durable Volume artifacts on FORCE_RELOAD reuse.
    """
    import json
    with open(gps_json) as _f:
        return {k: np.asarray(v, dtype=float) for k, v in json.load(_f).items()}


def _load_largest_model(sparse_dir):
    """Load the COLMAP model with the most images from a cluster's sparse dir."""
    import pycolmap
    sparse_dir = Path(sparse_dir)
    if not sparse_dir.exists():
        return None

    def _n(d):
        try:
            return len(pycolmap.Reconstruction(d).images)
        except Exception:
            return 0

    dirs = sorted(
        [d for d in sparse_dir.iterdir() if d.is_dir() and (d / "images.bin").exists()],
        key=_n, reverse=True,
    )
    return pycolmap.Reconstruction(dirs[0]) if dirs else None


def _apply_sim3(T, pts):
    s, R, t = T
    return s * (R @ np.atleast_2d(pts).T).T + t


def _place_clusters_shared_enu(cluster_models):
    """Place every cluster's COLMAP frame into ONE shared ENU frame and return the
    per-cluster Sim(3) transforms (COLMAP frame -> shared ENU metres) plus the shared
    GPS reference. The anchor cluster is georeferenced to GPS (Umeyama); each
    remaining cluster is tied to the already-placed set through its SHARED overlap
    cameras (same image names), falling back to its own GPS fit when it shares no
    cameras.

    This mirrors accumulate_orthomosaic's Passes A+B (the sparse projector). It is
    kept as a SEPARATE function so the GPU dense-cloud merge places its clusters in
    the SAME frame the sparse ortho uses, without modifying the validated sparse
    projector. (Unify with accumulate_orthomosaic when that path is next touched.)

    cluster_models: {cid: (sparse_dir, gps_json)}. Returns a dict with:
      T {cid: (scale, R, t)}, ref_lat/ref_lon/ref_alt, gps_tf, aligned (cids placed,
      largest first), n_tied, anchor, centers, gps, sizes.
    """
    import pycolmap

    centers = {}   # cid -> {name: np.array([x, y, z]) in that cluster's COLMAP frame}
    gps = {}       # cid -> {name: np.array([lat, lon, alt])}
    sizes = {}
    for cid, (sparse_dir, gps_json) in cluster_models.items():
        rec = _load_largest_model(sparse_dir)
        if rec is None or not rec.images:
            print(f"[place] cluster {cid}: no model — skipped", flush=True)
            continue
        c = {im.name: np.asarray(im.projection_center(), dtype=float)
             for _, im in rec.images.items() if im.has_pose}
        if len(c) < 3:
            print(f"[place] cluster {cid}: <3 posed images — skipped", flush=True)
            del rec
            gc.collect()
            continue
        centers[cid] = c
        gps[cid] = {k: v for k, v in _read_gps_priors(gps_json).items() if k in c}
        sizes[cid] = len(c)
        del rec
        gc.collect()
    if not centers:
        raise RuntimeError("No cluster produced a usable model.")
    order = sorted(centers, key=lambda c: sizes[c], reverse=True)

    # Shared global ENU reference (origin) = mean of ALL clusters' GPS priors.
    all_gps = {}
    for cid in centers:
        all_gps.update(gps[cid])
    if not all_gps:
        raise RuntimeError("No GPS priors across clusters — cannot georeference.")
    ref_lat, ref_lon, ref_alt = np.array(list(all_gps.values())).mean(0)
    gps_tf = pycolmap.GPSTransform()

    def _gps_fit(cid):
        names = [n for n in centers[cid] if n in gps[cid]]
        if len(names) < 3:
            return None
        src = np.array([centers[cid][n] for n in names])
        dst = np.array([gps_tf.ellipsoid_to_enu(np.array([gps[cid][n]]),
                                                 ref_lat, ref_lon, ref_alt)[0]
                        for n in names])
        return _umeyama_sim3(src, dst)

    # Anchor by GPS; tie neighbours through shared cameras; GPS-fallback the rest.
    T = {}
    placed_enu = {}   # image_name -> ENU centre (from already-placed clusters)
    n_tied = 0
    anchor = order[0]
    Ta = _gps_fit(anchor)
    if Ta is None:
        raise RuntimeError(
            f"Anchor cluster {anchor} lacks >=3 GPS priors for georeferencing."
        )
    T[anchor] = Ta
    for name, c in centers[anchor].items():
        placed_enu[name] = _apply_sim3(Ta, c)[0]

    remaining = [c for c in order if c != anchor]
    progress = True
    while remaining and progress:
        progress = False
        for cid in list(remaining):
            shared = [n for n in centers[cid] if n in placed_enu]
            if len(shared) >= 3:
                src = np.array([centers[cid][n] for n in shared])
                dst = np.array([placed_enu[n] for n in shared])
                T[cid] = _umeyama_sim3(src, dst)
                for name, c in centers[cid].items():
                    placed_enu.setdefault(name, _apply_sim3(T[cid], c)[0])
                remaining.remove(cid)
                progress = True
                n_tied += 1
                print(f"[place] cluster {cid}: tied via {len(shared)} shared cameras",
                      flush=True)
    for cid in list(remaining):  # not shared-camera-reachable -> own GPS fit
        Tg = _gps_fit(cid)
        if Tg is None:
            print(f"[place] cluster {cid}: no shared tie + <3 GPS — skipped", flush=True)
            continue
        T[cid] = Tg
        for name, c in centers[cid].items():
            placed_enu.setdefault(name, _apply_sim3(Tg, c)[0])
        print(f"[place] cluster {cid}: GPS fallback (no shared-camera tie)", flush=True)

    aligned = [cid for cid in order if cid in T]
    print(f"[place] {len(aligned)}/{len(centers)} clusters placed in shared ENU "
          f"({n_tied} via shared cameras, anchor {anchor})", flush=True)
    return {"T": T, "ref_lat": ref_lat, "ref_lon": ref_lon, "ref_alt": ref_alt,
            "gps_tf": gps_tf, "aligned": aligned, "n_tied": n_tied, "anchor": anchor,
            "centers": centers, "gps": gps, "sizes": sizes}


def accumulate_orthomosaic(cluster_models, img_dir, output_tiff, gsd_cm=3.0, max_workers=8, blend_gamma=4.0):
    """Blend per-cluster reconstructions into ONE georeferenced orthomosaic GeoTIFF.

    Clusters are placed in a single shared ENU frame: the largest cluster is
    georeferenced to GPS (Umeyama), and each remaining cluster is tied to the
    already-placed set through its SHARED overlap cameras (same image names) via a
    similarity fit on their projection centres — restoring cross-cluster
    consistency without a monolithic global bundle adjustment. Clusters with no
    shared-camera tie fall back to their own GPS fit. Each cluster projects onto
    its own local ground plane (z_med). One cluster reconstruction is held at a
    time; peak RAM = largest cluster + the canvas.
    """
    import pycolmap
    import rasterio
    import threading
    from concurrent.futures import ThreadPoolExecutor
    from rasterio.crs import CRS
    from rasterio.transform import from_origin

    t0 = time.perf_counter()
    img_dir = Path(img_dir)
    output_tiff = Path(output_tiff)

    # ---- Pass A: load each cluster once; collect projection centres + GPS (small) ----
    centers = {}   # cid -> {name: np.array([x,y,z]) in that cluster's COLMAP frame}
    gps = {}       # cid -> {name: np.array([lat,lon,alt])}
    sizes = {}
    for cid, (sparse_dir, gps_json) in cluster_models.items():
        rec = _load_largest_model(sparse_dir)
        if rec is None or not rec.images:
            print(f"[ortho] cluster {cid}: no model — skipped", flush=True)
            continue
        c = {im.name: np.asarray(im.projection_center(), dtype=float)
             for _, im in rec.images.items() if im.has_pose}
        if len(c) < 3:
            print(f"[ortho] cluster {cid}: <3 posed images — skipped", flush=True)
            del rec
            gc.collect()
            continue
        centers[cid] = c
        gps[cid] = {k: v for k, v in _read_gps_priors(gps_json).items() if k in c}
        sizes[cid] = len(c)
        del rec
        gc.collect()
    if not centers:
        raise RuntimeError("No cluster produced a usable model.")
    order = sorted(centers, key=lambda c: sizes[c], reverse=True)

    # ---- Shared global ENU reference (origin) ----
    all_gps = {}
    for cid in centers:
        all_gps.update(gps[cid])
    if not all_gps:
        raise RuntimeError("No GPS priors across clusters — cannot georeference.")
    ref_lat, ref_lon, ref_alt = np.array(list(all_gps.values())).mean(0)
    gps_tf = pycolmap.GPSTransform()

    def _gps_fit(cid):
        names = [n for n in centers[cid] if n in gps[cid]]
        if len(names) < 3:
            return None
        src = np.array([centers[cid][n] for n in names])
        dst = np.array([gps_tf.ellipsoid_to_enu(np.array([gps[cid][n]]),
                                                 ref_lat, ref_lon, ref_alt)[0] for n in names])
        return _umeyama_sim3(src, dst)

    # ---- Pass B: place clusters in ONE frame (anchor by GPS, neighbours by shared cameras) ----
    T = {}                # cid -> (scale, R, t): that cluster's COLMAP frame -> shared ENU
    placed_enu = {}       # image_name -> np.array ENU centre (from already-placed clusters)
    n_tied = 0

    anchor = order[0]
    Ta = _gps_fit(anchor)
    if Ta is None:
        raise RuntimeError(f"Anchor cluster {anchor} lacks >=3 GPS priors for georeferencing.")
    T[anchor] = Ta
    for name, c in centers[anchor].items():
        placed_enu[name] = _apply_sim3(Ta, c)[0]

    remaining = [c for c in order if c != anchor]
    progress = True
    while remaining and progress:
        progress = False
        for cid in list(remaining):
            shared = [n for n in centers[cid] if n in placed_enu]
            if len(shared) >= 3:
                src = np.array([centers[cid][n] for n in shared])
                dst = np.array([placed_enu[n] for n in shared])
                T[cid] = _umeyama_sim3(src, dst)
                for name, c in centers[cid].items():
                    placed_enu.setdefault(name, _apply_sim3(T[cid], c)[0])
                remaining.remove(cid)
                progress = True
                n_tied += 1
                print(f"[ortho] cluster {cid}: tied via {len(shared)} shared cameras", flush=True)
    for cid in list(remaining):  # GPS fallback for anything not shared-camera-reachable
        Tg = _gps_fit(cid)
        if Tg is None:
            print(f"[ortho] cluster {cid}: no shared tie + <3 GPS — skipped", flush=True)
            continue
        T[cid] = Tg
        for name, c in centers[cid].items():
            placed_enu.setdefault(name, _apply_sim3(Tg, c)[0])
        print(f"[ortho] cluster {cid}: GPS fallback (no shared-camera tie)", flush=True)

    aligned = [cid for cid in order if cid in T]
    print(f"[ortho] {len(aligned)}/{len(centers)} clusters aligned "
          f"({n_tied} via shared cameras, anchor {anchor} + {len(aligned)-1-n_tied} GPS)", flush=True)

    # ---- Pass C: union ground extent + per-cluster z_med (one reload per cluster) ----
    cluster_z = {}
    e_min = n_min = np.inf
    e_max = n_max = -np.inf
    for cid in aligned:
        rec = _load_largest_model(cluster_models[cid][0])
        if rec is None:
            continue
        Tc = T[cid]
        if rec.points3D:
            pe = _apply_sim3(Tc, np.array([p.xyz for p in rec.points3D.values()]))
            zc = float(np.median(pe[:, 2]))
            ground = pe[np.abs(pe[:, 2] - zc) < 20]
        else:
            cen = _apply_sim3(Tc, np.array([im.projection_center()
                                            for _, im in rec.images.items() if im.has_pose]))
            zc = float(cen[:, 2].mean() - 250)
            ground = cen
        cluster_z[cid] = zc
        e_min = min(e_min, ground[:, 0].min()); e_max = max(e_max, ground[:, 0].max())
        n_min = min(n_min, ground[:, 1].min()); n_max = max(n_max, ground[:, 1].max())
        del rec
        gc.collect()

    me = (e_max - e_min) * 0.05; mn = (n_max - n_min) * 0.05
    e_min -= me; e_max += me; n_min -= mn; n_max += mn
    gsd_m = gsd_cm / 100.0
    out_w = max(1, int((e_max - e_min) / gsd_m))
    out_h = max(1, int((n_max - n_min) / gsd_m))
    _canvas_gb = (out_h * out_w * 3 * 2 + out_h * out_w * 2) / 1024 ** 3
    if _canvas_gb > 4.0:
        print(f"[WARN] Canvas {_canvas_gb:.1f} GB > 4 GB — raise GSD_CM to reduce.", flush=True)
    print(f"Canvas: {out_w}x{out_h}px @ {gsd_cm:.1f}cm ({_canvas_gb:.2f} GB float16) "
          f"from {len(aligned)} cluster(s)", flush=True)

    canvas = np.zeros((out_h, out_w, 3), dtype=np.float16)
    weights = np.zeros((out_h, out_w), dtype=np.float16)
    _lock = threading.Lock()
    n_proj = [0]

    def _project_cluster(rec, T_cid, z_med):
        def _to_colmap(pe):
            s, R, t = T_cid
            return ((R.T @ (np.atleast_2d(pe) - t).T) / s).T

        def _one(kv):
            _, image = kv
            if not image.has_pose:
                return
            ip = img_dir / image.name
            if not ip.exists():
                return
            cam = rec.cameras[image.camera_id]
            f = cam.params[0]; cx = cam.params[1]; cy = cam.params[2]
            cw, ch = cam.width, cam.height
            cfw = image.cam_from_world(); R_cw = cfw.rotation.matrix(); t_cw = cfw.translation
            C_enu = _apply_sim3(T_cid, image.projection_center())
            h_above = float(C_enu[0, 2] - z_med)
            if h_above < 1.0:
                return
            hep = int(h_above * (cw / 2) / f / gsd_m) + 4
            hnp = int(h_above * (ch / 2) / f / gsd_m) + 4
            fc_px = (C_enu[0, 0] - e_min) / gsd_m
            fc_py = (n_max - C_enu[0, 1]) / gsd_m
            px0 = max(0, int(fc_px - hep)); px1 = min(out_w, int(fc_px + hep))
            py0 = max(0, int(fc_py - hnp)); py1 = min(out_h, int(fc_py + hnp))
            if px0 >= px1 or py0 >= py1:
                return
            with PILImage.open(ip) as pim:
                pim.draft("RGB", (cw, ch)); pim.load()
                if pim.width != cw or pim.height != ch:
                    pim = pim.resize((cw, ch), PILImage.BILINEAR)
                src = np.array(pim.convert("RGB"), dtype=np.float32)
            out_e = (e_min + (np.arange(px0, px1) + 0.5) * gsd_m).astype(np.float32)
            out_n = (n_max - (np.arange(py0, py1) + 0.5) * gsd_m).astype(np.float32)
            ge, gn = np.meshgrid(out_e, out_n); ht, wt = ge.shape
            P_enu = np.column_stack([ge.ravel(), gn.ravel(), np.full(ht * wt, z_med, np.float32)])
            P_cam = (R_cw.astype(np.float32) @ _to_colmap(P_enu).astype(np.float32).T
                     + t_cw.astype(np.float32)[:, None]).T
            valid = P_cam[:, 2] > 1e-6
            dz = np.where(valid, P_cam[:, 2], 1.0)
            u = np.where(valid, f * P_cam[:, 0] / dz + cx, -1.0).reshape(ht, wt)
            v = np.where(valid, f * P_cam[:, 1] / dz + cy, -1.0).reshape(ht, wt)
            in_img = (u >= 0) & (u < cw - 1) & (v >= 0) & (v < ch - 1)
            u0 = np.clip(u.astype(np.int32), 0, cw - 2); v0 = np.clip(v.astype(np.int32), 0, ch - 2)
            uf = (u - u0).clip(0, 1); vf = (v - v0).clip(0, 1)
            colors = (src[v0, u0] * ((1 - uf) * (1 - vf))[..., None]
                      + src[v0, u0 + 1] * (uf * (1 - vf))[..., None]
                      + src[v0 + 1, u0] * ((1 - uf) * vf)[..., None]
                      + src[v0 + 1, u0 + 1] * (uf * vf)[..., None])
            de = (u - cx) / (cw / 2); dn = (v - cy) / (ch / 2)
            w = np.maximum(0.0, 1.0 - np.sqrt(de ** 2 + dn ** 2)) ** blend_gamma * in_img
            with _lock:
                canvas[py0:py1, px0:px1] += (colors * w[..., None]).astype(np.float16)
                weights[py0:py1, px0:px1] += w.astype(np.float16)
                n_proj[0] += 1

        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            list(pool.map(_one, sorted(rec.images.items(), key=lambda kv: kv[1].name)))

    # ---- Pass D: reload each cluster and project into the shared canvas ----
    for cid in aligned:
        rec = _load_largest_model(cluster_models[cid][0])
        if rec is None:
            continue
        _project_cluster(rec, T[cid], cluster_z[cid])
        del rec
        gc.collect()
        print(f"[ortho] cluster {cid}: projected ({n_proj[0]} images total)", flush=True)

    has = weights > 0
    ortho = np.zeros((out_h, out_w, 3), dtype=np.uint8)
    ortho[has] = np.clip(canvas[has] / weights[has, None], 0, 255).astype(np.uint8)
    print(f"Coverage: {100.0 * has.sum() / (out_h * out_w):.1f}% ({n_proj[0]} images projected)")

    tl = gps_tf.enu_to_ellipsoid(np.array([[e_min, n_max, float(np.median(list(cluster_z.values())))]]),
                                 ref_lat, ref_lon, ref_alt)[0]
    dpm_lat = 1.0 / 111320.0
    dpm_lon = 1.0 / (111320.0 * np.cos(np.radians(ref_lat)))
    transform = from_origin(tl[1], tl[0], gsd_m * dpm_lon, gsd_m * dpm_lat)
    output_tiff.parent.mkdir(parents=True, exist_ok=True)
    import tempfile as _tf, shutil as _sh
    _local_tif = Path(_tf.gettempdir()) / output_tiff.name
    with rasterio.open(_local_tif, "w", driver="GTiff", height=out_h, width=out_w, count=3,
                       dtype="uint8", crs=CRS.from_epsg(4326), transform=transform, compress="lzw") as dst:
        for b in range(3):
            dst.write(ortho[:, :, b], b + 1)
    _sh.copy(str(_local_tif), str(output_tiff)); _local_tif.unlink()
    print(f"Orthomosaic written: {output_tiff} ({time.perf_counter() - t0:.0f}s)")
    return str(output_tiff)

In [ ]:
# ── Dense cloud → georeferenced products ───────────────────────────────────
# Two georeferencing frames:
#   * dense_clusters_to_products places EVERY cluster in ONE shared ENU frame
#     (anchor GPS + shared-camera ties, like the sparse accumulate_orthomosaic),
#     and writes per-cluster ortho/DSM AND a single co-registered merged ortho/DSM.
#     LAZ export uses the lidar_gbx DataSource writer in two phases:
#     phase 1 = sharded *_<group>_<cluster>.laz per-cluster parts;
#     phase 2 = dense_merged.laz folded from the parts (keepParts=True).
#   * dense_cloud_to_ortho / dense_cloud_to_laz keep the single-cluster path (a
#     cluster georeferenced to its OWN GPS mean) for probes / one-off use.
# Rasters (ortho/DSM) are EPSG:4326; LAZ clouds are reprojected to the local UTM
# zone (metric) so they carry a real projected CRS (tagged via the lidar_gbx crs
# option). Rasters go local-temp -> shutil.copy (UC FUSE + rasterio seek issues).


def _rasterize_enu_ortho(
    xe, ye, ze, r, g, b, ref_lat, ref_lon, ref_alt, gps_tf, out_ortho, out_dsm,
    gsd_cm=3.0,
):
    """Top-surface-rasterize an ENU-metre colored point cloud into a georeferenced
    ortho + DSM (EPSG:4326 GeoTIFFs). ref_* + gps_tf define the ENU origin used to
    convert the grid corner back to lon/lat (same equirectangular approximation as
    accumulate_orthomosaic)."""
    import shutil
    import tempfile
    from pathlib import Path

    import numpy as np
    import rasterio
    from rasterio.crs import CRS
    from rasterio.transform import from_origin

    from databricks.labs.gbx.pyrx.imagery import zbuffer_ortho

    # Robust to a thin/degenerate cloud: drop non-finite points, require a usable
    # count, and if the ground band (±20 m of median z) removes everything, fall
    # back to all finite points instead of raising on an empty array.
    finite = np.isfinite(xe) & np.isfinite(ye) & np.isfinite(ze)
    n_finite = int(finite.sum())
    if n_finite < 100:
        raise RuntimeError(
            f"_rasterize_enu_ortho: only {n_finite} finite points — reconstruction "
            "too sparse to rasterize; raise dense quality (DENSE_GEOM_CONSIST=True, "
            "more DENSE_SRC_IMAGES, larger DENSE_MAX_IMAGE_SIZE)."
        )
    xe, ye, ze = xe[finite], ye[finite], ze[finite]
    r, g, b = r[finite], g[finite], b[finite]
    z_med = float(np.median(ze))
    ground_mask = np.abs(ze - z_med) < 20.0
    if int(ground_mask.sum()) < max(10, int(0.01 * n_finite)):
        ground_mask = np.ones_like(ze, dtype=bool)  # band too thin — use all finite
    xe_g, ye_g = xe[ground_mask], ye[ground_mask]
    me = (xe_g.max() - xe_g.min()) * 0.05
    mn = (ye_g.max() - ye_g.min()) * 0.05
    e_min = xe_g.min() - me
    e_max = xe_g.max() + me
    n_min = ye_g.min() - mn
    n_max = ye_g.max() + mn
    gsd_m = gsd_cm / 100.0

    rgb, dsm, _ = zbuffer_ortho(
        xe, ye, ze, r, g, b,
        xmin=e_min, ymin=n_min, xmax=e_max, ymax=n_max, px=gsd_m,
    )

    # Georeference: same equirectangular approximation as accumulate_orthomosaic.
    tl = gps_tf.enu_to_ellipsoid(
        np.array([[e_min, n_max, z_med]]), ref_lat, ref_lon, ref_alt
    )[0]
    dpm_lat = 1.0 / 111320.0
    dpm_lon = 1.0 / (111320.0 * np.cos(np.radians(ref_lat)))
    geo_tf = from_origin(tl[1], tl[0], gsd_m * dpm_lon, gsd_m * dpm_lat)

    H, W = dsm.shape
    Path(out_ortho).parent.mkdir(parents=True, exist_ok=True)
    Path(out_dsm).parent.mkdir(parents=True, exist_ok=True)
    _tmp = Path(tempfile.gettempdir())
    _tmp_ortho = _tmp / Path(out_ortho).name
    _tmp_dsm = _tmp / Path(out_dsm).name

    with rasterio.open(
        _tmp_ortho, "w", driver="GTiff",
        height=H, width=W, count=3, dtype="uint8",
        crs=CRS.from_epsg(4326), transform=geo_tf, compress="lzw", nodata=0,
    ) as dst_f:
        for band in range(3):
            dst_f.write(rgb[band], band + 1)
    shutil.copy(str(_tmp_ortho), str(out_ortho))
    _tmp_ortho.unlink(missing_ok=True)

    with rasterio.open(
        _tmp_dsm, "w", driver="GTiff",
        height=H, width=W, count=1, dtype="float32",
        crs=CRS.from_epsg(4326), transform=geo_tf, compress="lzw",
    ) as dst_f:
        dst_f.write(dsm, 1)
    shutil.copy(str(_tmp_dsm), str(out_dsm))
    _tmp_dsm.unlink(missing_ok=True)

    print(f"Dense ortho → {out_ortho}  DSM → {out_dsm}  ({H}×{W}px @ {gsd_cm:.1f}cm)")
    return str(out_ortho), str(out_dsm)


def _enu_to_utm(xe, ye, ze, ref_lat, ref_lon):
    """Reproject shared-ENU-metre points to the local UTM zone so the exported LAZ
    carries a real *metric* projected CRS (a geographic CRS would force a coarse
    LAS scale in degrees). ENU→lon/lat uses the same equirectangular linearisation
    as the ortho georeferencing; lon/lat→UTM via pyproj. Returns
    (easting, northing, up, epsg)."""
    import numpy as np
    import pyproj

    xe = np.asarray(xe, dtype=float)
    ye = np.asarray(ye, dtype=float)
    dpm_lat = 1.0 / 111320.0
    dpm_lon = 1.0 / (111320.0 * np.cos(np.radians(ref_lat)))
    lon = ref_lon + xe * dpm_lon
    lat = ref_lat + ye * dpm_lat
    zone = int((ref_lon + 180.0) // 6.0) + 1
    epsg = (32600 if ref_lat >= 0 else 32700) + zone  # WGS84 UTM N/S
    tf = pyproj.Transformer.from_crs(4326, epsg, always_xy=True)
    east, north = tf.transform(lon, lat)
    return np.asarray(east), np.asarray(north), np.asarray(ze, dtype=float), epsg


def _export_enu_laz(xe, ye, ze, r, g, b, out_laz, crs=None):
    """Write a metric colored cloud to LAZ (tagged with `crs`) via write_xyzrgb_laz,
    local-temp then copy (laspy seeks to backfill the header → FUSE OSError on a
    direct Volume write). Returns the written path (suffix may fall back .laz→.las)."""
    import shutil
    import tempfile
    from pathlib import Path

    from databricks.labs.gbx.pyrx.imagery import write_xyzrgb_laz

    Path(out_laz).parent.mkdir(parents=True, exist_ok=True)
    _tmp_laz = str(Path(tempfile.gettempdir()) / Path(out_laz).name)
    _written = write_xyzrgb_laz(_tmp_laz, xe, ye, ze, r, g, b, crs=crs)
    _dst = str(Path(out_laz).with_suffix(Path(_written).suffix))
    shutil.copy(_written, _dst)
    Path(_written).unlink(missing_ok=True)
    print(f"Dense cloud LAZ → {_dst}  ({len(xe):,} points, crs={crs})")
    return _dst


def _write_dense_laz_dataset(spark, cluster_points, out_dir, crs=None):
    """cluster_points: list of (group, cluster, xe, ye, ze, r, g, b) numpy tuples.
    Assemble a points DataFrame, repartitionByRange by (group, cluster) to guarantee
    exactly one (group,cluster) key per partition, and write sharded
    *_<group>_<cluster>.laz via lidar_gbx."""
    from pyspark.sql import Row
    from databricks.labs.gbx.ds.lidar import LidarGbxDataSource

    try:
        spark.dataSource.register(LidarGbxDataSource)
    except Exception:
        pass
    n_parts = max(1, len(cluster_points))
    rows = []
    for (group, cluster, xe, ye, ze, r, g, b) in cluster_points:
        for i in range(len(xe)):
            rows.append(Row(x=float(xe[i]), y=float(ye[i]), z=float(ze[i]),
                            r=int(r[i]), g=int(g[i]), b=int(b[i]),
                            group=str(group), cluster=int(cluster)))
    df = spark.createDataFrame(rows)
    writer = (
        df.repartitionByRange(n_parts, "group", "cluster")
        .write.format("lidar_gbx")
        .option("groupCol", "group").option("clusterCol", "cluster")
        .option("fileName", "dense")
        .mode("overwrite")
    )
    if crs is not None:
        writer = writer.option("crs", str(crs))
    writer.save(out_dir)
    return out_dir


def _merge_dense_laz(spark, out_dir, file_name="dense_merged"):
    """Phase 2: fold the sharded *_<group>_<cluster>.laz parts under out_dir into
    one merged <file_name>.laz, KEEPING the per-cluster parts (keepParts). merge
    ignores the DataFrame rows (it folds the on-disk parts). The merge is
    driver-side (materialize-gated); on a large union it raises with a
    'run on classic' hint — expected, and the sharded parts remain the artifact."""
    from databricks.labs.gbx.ds.lidar import LidarGbxDataSource

    try:
        spark.dataSource.register(LidarGbxDataSource)
    except Exception:
        pass
    (
        spark.createDataFrame([(1,)], ["_"])  # merge ignores rows; folds on-disk parts
        .write.format("lidar_gbx")
        .option("merge", "true")
        .option("keepParts", "true")
        .option("fileName", file_name)
        .mode("append")
        .save(out_dir)
    )
    return out_dir


def _single_cluster_sim3(sparse_dir, gps_json):
    """Sim(3) (COLMAP frame → ENU metres) for ONE cluster, anchored to the mean of
    THAT cluster's GPS priors (the single-cluster analogue of the shared-frame fit
    in _place_clusters_shared_enu)."""
    import numpy as np
    import pycolmap

    gps_priors = _read_gps_priors(gps_json)
    rec = _load_largest_model(sparse_dir)
    if rec is None or not rec.images:
        raise RuntimeError("dense: no COLMAP model in sparse_dir")
    cam_centers = {
        im.name: np.asarray(im.projection_center(), dtype=float)
        for _, im in rec.images.items()
        if im.has_pose
    }
    gps_here = {k: v for k, v in gps_priors.items() if k in cam_centers}
    if len(gps_here) < 3:
        raise RuntimeError(f"dense: only {len(gps_here)} GPS priors — need >=3")
    ref_lat, ref_lon, ref_alt = np.array(list(gps_here.values())).mean(0)
    gps_tf = pycolmap.GPSTransform()
    names = [n for n in cam_centers if n in gps_here]
    src = np.array([cam_centers[n] for n in names])
    dst = np.array([
        gps_tf.ellipsoid_to_enu(np.array([gps_here[n]]), ref_lat, ref_lon, ref_alt)[0]
        for n in names
    ])
    return _umeyama_sim3(src, dst), ref_lat, ref_lon, ref_alt, gps_tf


def dense_cloud_to_ortho(ply_path, sparse_dir, gps_json, out_ortho, out_dsm, gsd_cm=3.0):
    """Single-cluster dense ortho + DSM, georeferenced to this cluster's own GPS mean.
    For the group deliverable use dense_clusters_to_products (shared ENU frame)."""
    import numpy as np

    from databricks.labs.gbx.pyrx.imagery import read_fused_ply

    cloud = read_fused_ply(str(ply_path))
    T, ref_lat, ref_lon, ref_alt, gps_tf = _single_cluster_sim3(sparse_dir, gps_json)
    pts = _apply_sim3(T, np.column_stack([cloud["x"], cloud["y"], cloud["z"]]))
    return _rasterize_enu_ortho(
        pts[:, 0], pts[:, 1], pts[:, 2], cloud["r"], cloud["g"], cloud["b"],
        ref_lat, ref_lon, ref_alt, gps_tf, out_ortho, out_dsm, gsd_cm,
    )


def dense_cloud_to_laz(ply_path, sparse_dir, gps_json, out_laz):
    """Single-cluster dense LAZ, reprojected to the cluster's local UTM zone (metric)."""
    import numpy as np

    from databricks.labs.gbx.pyrx.imagery import read_fused_ply

    cloud = read_fused_ply(str(ply_path))
    T, ref_lat, ref_lon, ref_alt, gps_tf = _single_cluster_sim3(sparse_dir, gps_json)
    pts = _apply_sim3(T, np.column_stack([cloud["x"], cloud["y"], cloud["z"]]))
    ux, uy, uz, epsg = _enu_to_utm(pts[:, 0], pts[:, 1], pts[:, 2], ref_lat, ref_lon)
    return _export_enu_laz(
        ux, uy, uz, cloud["r"], cloud["g"], cloud["b"], out_laz, crs=epsg
    )


def dense_clusters_to_products(
    cluster_models, dense_plys, *, merged_ortho, merged_dsm, merged_laz,
    cluster_paths, gsd_cm=3.0, group="all",
):
    """Place every cluster's fused.ply in ONE shared ENU frame (anchor GPS +
    shared-camera ties, mirroring accumulate_orthomosaic), then write:
      • per-cluster ortho/DSM  (cluster_paths[cid] → {'ortho', 'dsm'})
      • sharded per-cluster LAZ parts via lidar_gbx (phase 1)
      • dense_merged.laz merged from all parts via lidar_gbx (phase 2, keepParts)
      • one MERGED, co-registered ortho/DSM over all clusters (merged_ortho/dsm).

    Rasters are EPSG:4326; LAZ clouds are reprojected to the shared local UTM zone
    (metric CRS) and written via the lidar_gbx DataSource writer. cluster_models:
    {cid: (sparse_dir, gps_json)}; dense_plys: {cid: fused.ply}. Returns
    (used_cids, path_to_dense_merged_laz)."""
    import numpy as np
    from pathlib import Path

    from databricks.labs.gbx.pyrx.imagery import read_fused_ply

    placement = _place_clusters_shared_enu(cluster_models)
    ref_lat = placement["ref_lat"]
    ref_lon = placement["ref_lon"]
    ref_alt = placement["ref_alt"]
    gps_tf = placement["gps_tf"]

    agg = {k: [] for k in ("x", "y", "z", "r", "g", "b")}
    used = []
    used_points = []  # (group, cluster, ux, uy, uz, r, g, b) tuples for phase-1 lidar_gbx write
    last_epsg = None
    for cid in placement["aligned"]:
        ply = dense_plys.get(cid)
        if not ply or not Path(ply).exists():
            print(f"[dense-merge] cluster {cid}: no fused.ply — skipped", flush=True)
            continue
        try:
            cloud = read_fused_ply(str(ply))
            pts = _apply_sim3(
                placement["T"][cid],
                np.column_stack([cloud["x"], cloud["y"], cloud["z"]]),
            )
            xe, ye, ze = pts[:, 0], pts[:, 1], pts[:, 2]
            r, g, b = cloud["r"], cloud["g"], cloud["b"]
            cp = cluster_paths[cid]
            _rasterize_enu_ortho(
                xe, ye, ze, r, g, b, ref_lat, ref_lon, ref_alt, gps_tf,
                cp["ortho"], cp["dsm"], gsd_cm,
            )
            ux, uy, uz, epsg = _enu_to_utm(xe, ye, ze, ref_lat, ref_lon)
            last_epsg = epsg
            used_points.append((group, int(cid), ux, uy, uz, r, g, b))
            for k, v in zip(("x", "y", "z", "r", "g", "b"), (xe, ye, ze, r, g, b)):
                agg[k].append(v)
            used.append(cid)
        except Exception as e:
            print(f"[dense][skip] cluster {cid}: {e}", flush=True)
            continue
    if not used:
        raise RuntimeError("dense_clusters_to_products: no cluster produced points")

    X, Y, Z = (np.concatenate(agg[k]) for k in ("x", "y", "z"))
    R, G, B = (np.concatenate(agg[k]) for k in ("r", "g", "b"))
    print(f"[dense-merge] merging {len(used)} clusters → {len(X):,} points", flush=True)
    _rasterize_enu_ortho(
        X, Y, Z, R, G, B, ref_lat, ref_lon, ref_alt, gps_tf,
        merged_ortho, merged_dsm, gsd_cm,
    )
    # Phase 1: assemble all clusters' UTM points into a Spark DataFrame and write
    # sharded *_<group>_<cluster>.laz via the lidar_gbx DataSource writer.
    # repartitionByRange on (group, cluster) gives deterministic one-partition-per-
    # cluster isolation over the fixed key-set, so each shard carries exactly one
    # cluster's points and the filename is guaranteed correct.
    dense_laz_dir = str(Path(merged_laz).parent)
    _write_dense_laz_dataset(spark, used_points, dense_laz_dir, crs=last_epsg)
    # Phase 2 (best-effort): fold the sharded parts into dense_merged.laz.
    # The merge gate raises when the estimated cloud exceeds driver RAM (~256 MiB);
    # on failure the sharded parts (phase 1) are already written and remain readable
    # as a directory of parts via the lidar_gbx reader.
    try:
        _merge_dense_laz(spark, dense_laz_dir)
        _merged = str(Path(dense_laz_dir) / "dense_merged.laz")
    except Exception as e:
        print(f"[dense] phase-2 merge skipped ({e}); sharded parts remain the canonical artifact",
              flush=True)
        _merged = dense_laz_dir  # directory of parts — lidar_gbx reader accepts this
    return used, _merged

In [ ]:
# go-pmtiles CLI — download the pre-built Linux binary into a writable temp dir (Serverless env5 has a read-only system bin).
PMTILES_VERSION = "1.22.3"
_PMTILES_BIN_DIR = Path(tempfile.gettempdir()) / "gbx-bin"; _PMTILES_BIN_DIR.mkdir(parents=True, exist_ok=True); PMTILES_BIN = _PMTILES_BIN_DIR / "pmtiles"

if not PMTILES_BIN.exists():
    url = (
        f"https://github.com/protomaps/go-pmtiles/releases/download/v{PMTILES_VERSION}/"
        f"go-pmtiles_{PMTILES_VERSION}_Linux_x86_64.tar.gz"
    )
    with tempfile.TemporaryDirectory() as _tmp:
        tar = Path(_tmp) / "pmtiles.tar.gz"
        urllib.request.urlretrieve(url, tar)
        subprocess.run(["tar", "-xzf", str(tar), "-C", _tmp], check=True)
        shutil.copy2(Path(_tmp) / "pmtiles", PMTILES_BIN)
        PMTILES_BIN.chmod(PMTILES_BIN.stat().st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    print(f"Installed pmtiles → {PMTILES_BIN}")
else:
    print(f"pmtiles present: {PMTILES_BIN}")

print(subprocess.run([str(PMTILES_BIN), "version"], capture_output=True, text=True).stdout.strip())

In [ ]:
def count_registered(model_dir) -> int:
    """Count images registered in a COLMAP model dir (binary or text format)."""
    try:
        rec = pycolmap.Reconstruction(Path(model_dir))
        return len(rec.images)
    except Exception:
        return 0


def is_model_dir(d) -> bool:
    """True if directory contains a COLMAP model (binary or text)."""
    d = Path(d)
    return d.is_dir() and ((d / "images.bin").exists() or (d / "images.txt").exists())